<a href="https://colab.research.google.com/github/Seledalia/TPs-genomica-/blob/main/Copia_de_Assembly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Trabajo Práctico
# Ensamblaje genómico

Objetivos


*   Familiarizarse con el flujo de trabajo para realizar un montaje genómico
*   Entender diferencias entre las diferentes aproximaciones para montar genomas
*   Diferenciar montajes de lecturas cortas (simples y pareadas) y largas.







## Primera parte: Ensamblaje de lecturas cortas

En diciembre de 2019 se notificaron en la provincia de Wuhan, China, varios casos de neumonía viral causada por un agente etiológico desconocido.
A partir de una muestra clínica de un paciente, se purificó ADN y se realizó la secuenciación mediante un equipo Illumina. Los datos generados fueron depositados en la base de datos Sequence Read Archive (SRA) bajo el número de acceso SRR10971381.

En este primer ejercicio, realizaremos un montaje genómico de esta muestra, para identificar el patógeno desconocido.

### 0. Preparación de ambientes (~5 min)

In [ ]:
# Descargar micromamba
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba

# Crear entorno llamado "colab" con numpy
!./bin/micromamba create -y -n assembly -c conda-forge -c bioconda fastqc seqkit cutadapt minia spades taxonkit blast goldrush

# Agregar conda al ambiente
import os
os.environ['PATH'] = '/content/bin/:' + os.environ['PATH']


In [ ]:
# Instalar y configurar sra-tools
!apt-get update
!apt-get install -y sra-toolkit

!mkdir -p ~/.ncbi

# setear SRA cache location
!micromamba run -n assembly  vdb-config -s /repository/user/main/public/root=/content/sra

# crear cache
!mkdir -p /content/sra


### 1. Obtención de las lecturas y limpieza de los datos


Los datos de secuenciación (ID SRR10971381) están disponibles en la base de datos SRA.  

Para descargarlos, se utiliza fastq-dump de SRA Toolkit.

Se obtienen las primeras 100000 lecturas del experimento paired-end usando la opción **--split-files**, que genera archivos *forward* y *reverse* separados. Los resultados se guardan en el directorio **reads**.

In [ ]:
# Crea el directorio para
# Crea el directorio temporal para las lecturas
!mkdir reads

# Obtiene los datos.
num_reads = 100000

!fastq-dump -X {num_reads} -A SRR10971381 --split-files --gzip --outdir reads

In [ ]:
# ver si las lecturas fueron descargadas
!ls reads

In [ ]:
# Variables con el path a las lecturas
F1 = "reads/SRR10971381_1.fastq.gz"
F2 = "reads/SRR10971381_2.fastq.gz"

In [ ]:
# Control de calidad de las lecturas.
# Una vez que las descargamos, veo si el archivo es correcto y tiene la cantidad de lecturas correctas:

!micromamba run -n assembly seqkit stats {F1} {F2}


In [ ]:
# Continuacion
# Corre FastQC con los datos descargados para visualizar sus estadísticas de calidad


!micromamba run -n assembly fastqc {F1}
!micromamba run -n assembly fastqc {F2}

Abro en el navegador los archivos .html para ver la calidad que se encuentran en la carpeta /reads.


En este caso vemos que muchas lecturas son de muy mala calidad,
qué deberíamos hacer? 🤔

El ***trimming*** en genómica es el proceso de eliminar secuencias de baja calidad, adaptadores y bases ambiguas de las lecturas crudas obtenidas por secuenciación. Este paso mejora la precisión del análisis posterior, como el ensamblaje o la alineación, al conservar únicamente las porciones confiables de cada lectura.

In [ ]:
# Remover adaptadores, remover secuencias debaja calidad, remover reads sin pares

# Establecer parámetros
F1trim = "reads/SRR10971381_trimmed_1.fastq.gz"
F2trim = "reads/SRR10971381_trimmed_2.fastq.gz"

ADAPTER_FWD = "AGATCGGAAGAGCACACGTCTGAACTCCAGTCA"
ADAPTER_REV = "AGATCGGAAGAGCGTCGTGTAGGGAAAGAGTGT"

# Tamaño mínimo
minlen = 50

# Calidad mínima en extremos (corta extremos con menor calidad) codificado en phred score
qual = 20

# correr cutadapt para realizar la limpieza
!micromamba run -n assembly cutadapt -a {ADAPTER_FWD} -A {ADAPTER_REV} -o {F1trim} -p {F2trim} {F1} {F2} --minimum-length {minlen} -q {qual}

In [ ]:
# Visualizo nuevamente la calidad de las lecturas ya recortadas:
!micromamba run -n assembly seqkit stats {F1trim} {F2trim}


### 2. Ensamblaje


Existen diversas herramientas para el ensamblaje de novo de genomas, como Minia, SPAdes y Velvet, entre otras.

En este caso, realizaremos el ensamblaje utilizando Minia y SPAdes. Ambos son ensambladores basados en *de Bruijn graphs*

Minia es un ensamblador liviano que no utiliza información sobre pareamiento de lecturas durante el ensamblaje.

Por el contrario, SPAdes está optimizado para genomas bacterianos y metagenomas pequeños, y suele producir ensamblajes de alta calidad gracias a su estrategia de múltiples k-mers y por llevar en cuenta informacion sobre lecturas pareadas.



#### Minia

In [ ]:
# crear carpeta para organizar resultados en nuestro drive
miniadir = "resultados/minia"
!mkdir resultados
!mkdir {miniadir}

resultados = miniadir + "/minia" # path + prefijo

# listar archivos fastq para usarlos en minia
!ls reads/*trimmed* > read_list


In [ ]:
!micromamba run -n assembly minia -in read_list -out {resultados} -kmer-size 75

#### SPAdes

**Importante** ⏰ Este paso demora aprox. ~15 minutos. Aprovechamos para hacer el intervalo.  😴

In [ ]:
spadesdir =  "resultados/spades"

!mkdir {spadesdir}


In [ ]:
spadesdir = "resultados/spades"
!mkdir -p {spadesdir}

In [ ]:
!micromamba run -n assembly spades.py -1 {F1trim} -2 {F2trim} -o {spadesdir} -k 21,33,55,75

### Evaluación de calidad de ensamblajes  

Comparar ambos ensamblajes para ver el rendimiento de ambas aproximaciones.

In [ ]:
# Qué archivos se generaron en el ensamblaje con minia? # qué archivo contiene el assembly?
Minia genera varios archivos pero el más importante es minia.contigs.fa — ese es el assembly. Contiene los contigs ensamblados directamente a partir de los k-mers


In [ ]:
# incluir el assembly dentro de una variable
minia_res = miniadir + "/minia.contigs.fa"


In [ ]:
# Qué archivos se generaron en el ensamblaje con spades? # qué archivo contiene el assembly?
SPAdes genera bastantes archivos, los más importantes son:

contigs.fasta — contigs ensamblados
scaffolds.fasta — scaffolds (el assembly final)

El que se usa como resultado final es scaffolds.fasta.



In [ ]:
# qué archivo contiene el assembly? contigs.fasta o scaffolds.fasta ? cual es la diferencia?

Diferencia entre contigs.fasta y scaffolds.fasta:

Contigs son secuencias continuas ensambladas sin interrupciones — SPAdes une lecturas que se solapan directamente.
Scaffolds son contigs unidos usando la información de las lecturas pareadas (paired-end).
Cuando SPAdes sabe que dos contigs están cerca porque hay lecturas que los conectan, los une con una cadena de N en el medio representando la región desconocida.



In [ ]:
# incluir el ensamblaje de spades dentro de una variable
spades_res = spadesdir + "/scaffolds.fasta"


In [ ]:
# verificamos la calidad de los ensamblajes con seqkit

!micromamba run -n assembly seqkit stats {minia_res} {spades_res} --all

In [ ]:
# limpiaremos el directorio de lecturas para liberar espacio
!rm -r reads

## Segunda parte: Ensamblaje de lecturas largas

El ensamblaje de lecturas largas suelen basarse en algoritmos de solapamiento (Overlap-Layout-Consensus, OLC), que comparan todas las lecturas entre sí para construir un grafico de solapamientos y generar el consenso final del genoma.

**Goldrush** es un ensamblador de lecturas largas que emplea un enfoque optimizado optimizado para recursos computacionales limitados: el ensamblador construye primero un *golden path* (un subconjunto representativo de lecturas equivalentes a 1X de cobertura) sin crear un gráfico global de solapamientos ni de k-mers, y evita el paso intensivo que representa la comparación de todas las lecturas contra todas las lecturas.

Luego, en pasos posteriores, puede refinar y extender esas secuencias (p. ej., corrección y *scaffolding*), pero no produce un gráfico explícito tipo OLC o de Bruijn como salida principal.

En este ejercicio utilizaremos goldrush para montar el genoma de una muestra (SRR31637353) de un paciente de COVID, secuenciada con nanopore R9.

**Importante** ❗  
En este ejercicio ignoraremos el paso de remoción de adaptadores con fines didacticos. En la realidad, este paso no debe ser ignorado.


In [ ]:
# Crea el directorio para las lecturas y el assembly
!mkdir long_reads

# Obtiene los datos. Descargaremos el mismo número de lecturas

num_reads = 100000

!fastq-dump -X {num_reads} -A SRR31637353 --gzip --outdir long_reads

In [ ]:
# definir parametros para assembly
#1. tamaño genómico esperado, supongamos 100Kb
GS = 100000
#2. número de procesadores a utilizar
threads = 12

# ingresar al la carpeta
%cd long_reads

# descomprimir lecturas
!gunzip SRR31637353.fastq.gz

In [ ]:
# montar el genoma
!micromamba  run -n assembly goldrush path-polish reads=SRR31637353 G=$GS t=$threads

In [ ]:
# Verificar que archivos fueron creados




In [ ]:
# Cuál es la calidad del ensamblaje?




In [ ]:
# volver al directorio base donde estabamos trabajando
%cd /content

**Preguntas ...**
1. Si usamos el mismo número de lecturas de illumina que de nanopore, por qué los ensamblajes son tan diferentes en terminos de calidad? 🤔
2. Por qué al usar goldrush no definimos el tamaño de kmer?
3. Explorar el github de goldrush (https://github.com/BirolLab/goldrush) e identifique qué pasos del ensamblaje realizamos con ese comando (goldrush path-polish). Podríamos haber ejecutado otros pasos?

# Tercera parte: Identificar la composición de la muestra

Realizaremos una búsqueda de blast contra una base de datos de genomas virales. Esto se realizará únicamente para el montaje de SPAdes, pero pueden repetirlo para los demas montajes.



In [ ]:
# descargar base de datos de virus de blast
!wget https://ftp.ncbi.nlm.nih.gov/blast/db/ref_viruses_rep_genomes.tar.gz
!micromamba run -n assembly update_blastdb.pl --decompress ref_viruses_rep_genomes

# descargar datos sobre taxonomia de blast
!wget ftp://ftp.ncbi.nih.gov/pub/taxonomy/taxdump.tar.gz
!tar -xzvf taxdump.tar.gz
!mkdir /root/
!mkdir /root/.taxonkit

!cp names.dmp nodes.dmp delnodes.dmp merged.dmp /root/.taxonkit

In [ ]:
# correr blast local
!micromamba run -n assembly blastn -db ref_viruses_rep_genomes -query {spades_res} -outfmt 6 > blast.txt

In [ ]:
# quedarme con únicamente el mejor hit por secuencia
!sort -k1,1 -k12,12g blast.txt | awk '!seen[$1]++' > best_hits_per_contig.tsv

# generar tabla intermedia con taxonomia de los hits
!cut -f2  best_hits_per_contig.tsv > acc.list
!micromamba run -n assembly blastdbcmd -db ref_viruses_rep_genomes -entry_batch acc.list -outfmt "%a %T" | awk '{print $2}' > acc_taxid.tsv

# generar conteos por familia
!micromamba run -n assembly taxonkit lineage acc_taxid.tsv | micromamba run -n assembly taxonkit reformat -r "Unclassified" -f "{f}"  > results.taxonomy


In [ ]:
# Script en python para graficar los resultados

import pandas as pd, matplotlib.pyplot as plt

# leer resultados
df = pd.read_csv("results.taxonomy", sep=r"\s+", header=None, names=["count","taxonomy"], engine="python", usecols=[0,1], dtype=str)
df = df["taxonomy"].str.split(";", expand=True)

# renomabrar algunas columnas
df = df.rename(columns={1:"realm",2:"kingdom",3:"phylum",4:"subphylum",5:"order"})

# contar los hits por columna seleccionada y plotear
df.groupby("phylum").count()[0].plot.bar()

**Al finalizar responda:** 🤔


1.   Cuáles fueron los principales microorganismos presentes en la muestra?
2.   El último script puede ser modificado para contar el número de hits en otros niveles taxonómicos. En niveles taxonómicos inferiores, hay algun grupo que se vuelva relevante? (Crear celdas de código nuevo para responder esta pregunta)
3.   Estos resultados son consistentes por una infección por coronavirus?
4.   Discuta potenciales sesgos de este análisis. Podría diagnosticar en base a este resultado?